# Frontier LLM Research Engineering — Phase 0 scaled pilots

Run on Colab with a GPU runtime (**A100** preferred; **L4** for the FP8 pilot; T4 for the free-GPU
checks). Run the cells top to bottom. Every pilot writes to Google Drive and resumes if the session
disconnects: just run the same cell again.

Order matters when compute units run out: **P1 (noise floor) first**, then the others.

In [ ]:
# 1. Mount Drive (outputs survive disconnects) and choose where pilots write.
from google.colab import drive
drive.mount('/content/drive')
PILOT_DIR = '/content/drive/MyDrive/frontier-llm-pilots'
import os; os.makedirs(PILOT_DIR, exist_ok=True); os.environ['PILOT_DIR'] = PILOT_DIR
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2. Get the course code. Option A: the GitHub repo (set REPO_URL; a private repo needs a token in the URL).
#    Option B: upload a zip of the repo to Drive and set ZIP_PATH.
REPO_URL = 'https://github.com/tal-giladi/frontier-llm-research-course.git'
ZIP_PATH = ''   # e.g. '/content/drive/MyDrive/frontier-llm-research-course.zip'
import os
if ZIP_PATH:
    !rm -rf /content/course && mkdir -p /content/course && unzip -q "$ZIP_PATH" -d /content/course
    ROOT = '/content/course/' + os.listdir('/content/course')[0] if len(os.listdir('/content/course')) == 1 else '/content/course'
else:
    !rm -rf /content/course && git clone -q "$REPO_URL" /content/course
    ROOT = '/content/course'
%cd $ROOT
!git log --oneline -1 2>/dev/null || true

In [ ]:
# 3. Install the pinned stack (references/versions.md). PyTorch 2.14.1 CUDA wheel, then the lab package.
!pip -q install torch==2.14.1 --index-url https://download.pytorch.org/whl/cu128
!pip -q install numpy==2.5.3 scipy transformers==5.18.0 safetensors tokenizers==0.23.2 datasets==5.0.1 matplotlib==3.11.2 pytest==9.1.1 pyyaml==6.0.3 torchao==0.18.0
!pip -q install -e labs/common
import torch; print(torch.__version__, torch.cuda.get_device_name(0))
!python -m pytest labs/common/tests/test_model.py -x -q 2>&1 | tail -3   # quick check; the full suite runs on CPU in the build

In [ ]:
# 4. Data-v0 at pilot size (vocab 32768). ~800k documents is ~0.8B train tokens: enough for the P1 ladder.
#    Writes to Drive; skipped if already there. Expect roughly 30-60 minutes and ~2 GB.
DATA = f'{PILOT_DIR}/data-v0-32k'
import os
if not os.path.exists(f'{DATA}/meta.json'):
    !python -m frontierlab.data.prepare --docs 800000 --vocab 32768 --out "$DATA"
# Lab data and every runs/ folder live on Drive, so a disconnect or a new session loses nothing.
!mkdir -p "$PILOT_DIR/labdata" "$PILOT_DIR/runs" && rm -rf labs/common/data runs
!ln -sfn "$PILOT_DIR/labdata" labs/common/data && ln -sfn "$PILOT_DIR/runs" runs
!rm -rf labs/common/data/v0 && ln -sfn "$DATA" labs/common/data/v0
!cat "$DATA/meta.json" | head -40

## P1 — Module 1: noise floor on a scale ladder (highest priority)

Question: how does the seed standard deviation of held-out loss scale with model size, compared with
typical architecture effects? Ladder: `pilot-10m`, `pilot-30m`, `pilot-70m` × 3 seeds, tokens ≈ 10 ×
non-embedding parameters (shorter than compute-optimal, to fit the budget — the summary says so).
Approximate A100 time: 10m ≈ 3×6 min, 30m ≈ 3×25 min, 70m ≈ 3×60 min (PROJECTED; the run logs record
the measured tokens/s and MFU). Runs checkpoint every 200 steps; after a disconnect rerun the setup
cells and this cell — finished runs are skipped and unfinished ones resume exactly.

In [ ]:
import json, os, subprocess
LADDER = {'pilot-10m': 9.4e6, 'pilot-30m': 31.5e6, 'pilot-70m': 70.8e6}
B, T, ACC = 32, 1024, 1                      # 32k tokens per step
PEAK = 'A100' if 'A100' in torch.cuda.get_device_name(0) else ('L4' if 'L4' in torch.cuda.get_device_name(0) else None)
for preset, n in LADDER.items():
    steps = int(10 * n / (B * T * ACC))
    for seed in (0, 1, 2):
        run = f'{PILOT_DIR}/p1/{preset}-s{seed}'
        if os.path.exists(f'{run}/metrics.jsonl') and any('"split": "val"' in l and f'"step": {steps}' in l for l in open(f'{run}/metrics.jsonl')):
            print('done', run); continue
        cmd = ['python', '-m', 'frontierlab.train.loop', '--run', run, '--data', DATA, '--preset', preset,
               '--steps', str(steps), '--batch', str(B), '--seq', str(T), '--grad-accum', str(ACC),
               '--lr', '3e-3', '--warmup', str(max(20, steps // 50)), '--dtype', 'bf16', '--seed', str(seed),
               '--eval-every', str(max(50, steps // 5)), '--eval-windows', '128', '--ckpt-every', '200',
               '--log-every', '20', '--question', 'P1 noise floor']
        if PEAK: cmd += ['--peak', PEAK]
        print(' '.join(cmd)); subprocess.run(cmd, check=True)

In [ ]:
# P1 summary: final val loss per seed, seed std per size, measured tok/s and MFU.
import json, glob, statistics as st
rows = {}
for f in sorted(glob.glob(f'{PILOT_DIR}/p1/*/metrics.jsonl')):
    name = f.split('/')[-2]; preset, seed = name.rsplit('-s', 1)
    recs = [json.loads(l) for l in open(f)]
    val = [r for r in recs if r['split'] == 'val']; tr = [r for r in recs if r['split'] == 'train']
    rows.setdefault(preset, []).append((val[-1]['loss'] if val else None, st.median(r['tok_per_s'] for r in tr[-20:]),
                                         st.median(r.get('mfu', 0) for r in tr[-20:])))
summary = {}
for preset, v in rows.items():
    losses = [x[0] for x in v if x[0] is not None]
    summary[preset] = {'seeds': len(losses), 'val_losses': losses,
                       'seed_std': st.stdev(losses) if len(losses) > 1 else None,
                       'tok_per_s': [x[1] for x in v], 'mfu': [x[2] for x in v]}
print(json.dumps(summary, indent=2))
os.makedirs(f'{PILOT_DIR}/summary', exist_ok=True)
json.dump(summary, open(f'{PILOT_DIR}/summary/p1.json', 'w'), indent=2)

In [ ]:
# Last cell: marks P1 complete when no command of it has failed; the status notebook reads this.
import glob, os
failed = glob.glob(f'{PILOT_DIR}/.failed/P1-*')
if failed:
    print('P1 NOT complete; failed commands (rerun this notebook to retry):', failed)
else:
    open(f'{PILOT_DIR}/.done/P1-COMPLETE', 'w').write('ok')
    print('P1 complete')